In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q20/annotated/checkpoints/post_cell_4.pickle

trying: ['DATA_ROOT']
me:  0
trying: ['lineitem']


me:  1
trying: ['tpch_parent']
me:  0
trying: ['load_lineitem']
me:  1
trying: ['load_supplier']
me:  9
trying: ['part']
me:  3
trying: ['supplier']
me:  9
trying: ['load_part']
me:  3
trying: ['pd']
me:  0
trying: ['orig_output']
me:  2
trying: ['load_partsupp']
me:  7
trying: ['load_nation']
me:  5
trying: ['partsupp']
me:  7
trying: ['nation']
me:  5
trying: ['STORAGE_OPTS']
me:  0


Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable orig_output
Declaring variable part
Declaring variable load_part
Declaring variable load_nation
Declaring variable nation
Declaring variable load_partsupp
Declaring variable partsupp
Declaring variable load_supplier
Declaring variable supplier


In [4]:
%%RecordEvent
%%time
### cell 5 ###
# Filter predicates using GPU‐friendly operations (no pd.Timestamp scalars)
psel = part.P_NAME.str.startswith("azure")
nsel = nation.N_NAME == "JORDAN"
lsel = (
    lineitem.L_SHIPDATE >= "1996-01-01"
) & (
    lineitem.L_SHIPDATE < "1997-01-01"
)

fpart = part[psel]
fnation = nation[nsel]
flineitem = lineitem[lsel]

jn1 = fpart.merge(partsupp, left_on="P_PARTKEY", right_on="PS_PARTKEY")
jn2 = jn1.merge(
    flineitem,
    left_on=["PS_PARTKEY", "PS_SUPPKEY"],
    right_on=["L_PARTKEY", "L_SUPPKEY"],
)

gb = jn2.groupby(
    ["PS_PARTKEY", "PS_SUPPKEY", "PS_AVAILQTY"],
    as_index=False,
    sort=False
)["L_QUANTITY"].sum()

# retain only those with available quantity > half of shipped quantity
gbsel = gb.PS_AVAILQTY > (0.5 * gb.L_QUANTITY)
fgb = gb[gbsel]

jn3 = fgb.merge(supplier, left_on="PS_SUPPKEY", right_on="S_SUPPKEY")
jn4 = fnation.merge(jn3, left_on="N_NATIONKEY", right_on="S_NATIONKEY")[["S_NAME", "S_ADDRESS"]]

total = jn4.sort_values("S_NAME").drop_duplicates()

CPU times: user 102 ms, sys: 19.1 ms, total: 121 ms
Wall time: 144 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q20/rewritten/q20_rewrite/checkpoints/post_cell_5_try_0.pickle

migration speed (bps): 789381002.9687302
---------------------------
variables to migrate:
flineitem 48
nsel 48
tpch_parent 54
jn2 48
load_lineitem 144
STORAGE_OPTS 64
lineitem 48
fnation 48
load_nation 144
pd 72
nation 48
jn4 48
jn3 48
orig_output 16
partsupp 48
gbsel 48
load_partsupp 144
psel 48
total 48
load_supplier 144
supplier 48
lsel 48
fpart 48
jn1 48
fgb 48
DATA_ROOT 80
load_part 144
part 48
gb 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q20/rewritten/q20_rewrite/checkpoints/post_cell_5_try_0.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['part']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['nation']
Future variables ['part', 'lineitem']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['partsupp']
Future variables ['part', 'nation', 'lineitem']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables ['supplier']
Future variables ['part', 'nation', 'lineitem', 'partsupp']
Modified dataframes
======= Cell 5 =======
Input variables []
Active variables []
Intermediate variables ['lsel', 'fpart', 'gb', 'fgb', 'fnation', 'flineitem', 'total', 'jn4', 'psel', 'nsel', 'jn3', 'jn1', 'jn2', 'gbsel']
Future variables [

In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q20/opt_cell_exec_info_5_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[5], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q20/annotated/checkpoints/post_cell_5.pickle
assert compare_df(total_opt, total)

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
